In [3]:
# ==============================
# 整合最终版（修复权重对比图）：美股蓝筹再平衡工具
# 双模式：风险平价+IEF | 固定权重 | 完整回测 | 交互界面
# ==============================

# 安装依赖（Colab首次运行需要）
#!pip install -q yfinance numpy pandas matplotlib ipywidgets scipy plotly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import yfinance as yf
from ipywidgets import interact, Dropdown, IntSlider, Checkbox
from scipy.optimize import minimize
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==============================
# 全局配置（所有参数集中管理）
# ==============================
# 资产池
STOCKS_ONLY = ['AAPL', 'MSFT', 'JNJ', 'PG', 'JPM', 'XOM']
STOCKS_PLUS_IEF = ['AAPL', 'MSFT', 'JNJ', 'PG', 'JPM', 'XOM', 'IEF']
STOCK_COUNT = 6
BOND_WEIGHT = 0.1  # 国债固定占比10%
STOCK_WEIGHT = 0.9  # 股票总占比90%

# 你的固定权重配置（来自untitled30.py）
FIXED_WEIGHTS_STOCKS = pd.Series({
    "AAPL": 0.20, "MSFT": 0.20, "JNJ": 0.15, "PG": 0.15, "JPM": 0.15, "XOM": 0.15
})

# 三档风险模式参数
MODES = {
    '保守型': {
        'annual_return': 0.0035,
        'max_drawdown': -0.0009,
        'base_threshold': 0.02,
        'description': '极低风险，几乎不亏'
    },
    '平衡型': {
        'annual_return': 0.009,
        'max_drawdown': -0.0045,
        'base_threshold': 0.05,
        'description': '小幅波动，收益稳健'
    },
    '进取型': {
        'annual_return': 0.021,
        'max_drawdown': -0.028,
        'base_threshold': 0.08,
        'description': '适度波动，收益更高'
    }
}

# 交易成本参数
COMMISSION_RATE = 0.001  # 千分之一佣金
SLIPPAGE_RATE = 0.0005   # 万分之五滑点

# ==============================
# 1. 扩展的真实数据获取（支持回测长周期）
# ==============================
def get_yahoo_finance_data(assets, period='3mo', start_date=None, end_date=None):
    """
    从雅虎财经获取数据：
    - 支持短周期（用于实时权重计算）
    - 支持长周期（用于历史回测）
    """
    try:
        if start_date and end_date:
            data = yf.download(
                tickers=assets + ['^VIX'],
                start=start_date,
                end=end_date,
                auto_adjust=False,
                progress=False
            )['Adj Close']
        else:
            data = yf.download(
                tickers=assets + ['^VIX'],
                period=period,
                interval='1d',
                auto_adjust=False,
                progress=False
            )['Adj Close']

        if data.isnull().any().any():
            data = data.dropna()

        if len(data) < 20:
            raise ValueError("历史数据不足")

        latest_prices = data[assets].iloc[-1].values
        latest_vix = data['^VIX'].iloc[-1]
        returns = data[assets].pct_change().dropna()
        cov_matrix = returns.cov().values * 252

        print(f"✅ 成功从雅虎财经获取数据")
        print(f"📅 数据期间：{data.index[0].date()} 至 {data.index[-1].date()}")

        return latest_prices, latest_vix, cov_matrix, returns, data

    except Exception as e:
        print(f"❌ 无法获取数据：{str(e)}")
        raise

# ==============================
# 2. 专业风险平价算法（scipy优化器版）
# ==============================
def calculate_risk_parity_weights(cov_matrix):
    n_assets = cov_matrix.shape[0]
    initial_weights = np.ones(n_assets) / n_assets
    constraints = (
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'ineq', 'fun': lambda w: w}
    )

    def risk_parity_objective(weights):
        portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        marginal_risk = np.dot(cov_matrix, weights) / portfolio_vol
        risk_contrib = weights * marginal_risk
        return np.sum((risk_contrib - risk_contrib.mean()) ** 2)

    result = minimize(
        risk_parity_objective,
        initial_weights,
        method='SLSQP',
        constraints=constraints,
        options={'maxiter': 1000, 'disp': False}
    )

    if not result.success:
        print(f"⚠️  风险平价优化未完全收敛，使用等权")
        return np.ones(n_assets) / n_assets

    return result.x

# ==============================
# 3. 交易成本计算
# ==============================
def calculate_trading_costs(trades):
    total_cost = 0.0
    for amt in trades.values():
        total_cost += abs(amt) * (COMMISSION_RATE + SLIPPAGE_RATE)
    return round(total_cost, 2)

# ==============================
# 4. 动态再平衡阈值
# ==============================
def dynamic_threshold(base_th, vix):
    if vix > 25:
        return base_th * 1.5
    elif vix < 15:
        return base_th * 0.75
    return base_th

# ==============================
# 5. 再平衡计算（修复：返回pandas Series权重）
# ==============================
def calc_rebalance(prices, holdings, target_weights, threshold, assets):
    total_value = np.sum(prices * holdings)
    # 转换为pandas Series，确保索引与资产一致
    current_weights = pd.Series((prices * holdings) / total_value, index=assets)
    target_weights_series = pd.Series(target_weights, index=assets)
    trades = {}

    for i, asset in enumerate(assets):
        diff = current_weights[i] - target_weights_series[i]
        if abs(diff) > threshold:
            trade_amount = (target_weights_series[i] - current_weights[i]) * total_value
            trades[asset] = round(trade_amount, 2)

    return trades, current_weights, target_weights_series, total_value

# ==============================
# 6. 5年收益模拟
# ==============================
def simulate_5year(principal, annual_return, leverage=1):
    annual_return_leveraged = annual_return * leverage
    months = np.arange(61)
    values = [principal]

    for _ in months[1:]:
        monthly_return = annual_return_leveraged / 12
        values.append(values[-1] * (1 + monthly_return))

    return months, values

# ==============================
# 7. 杠杆爆仓风险计算
# ==============================
def calculate_liquidation_risk(principal, max_drawdown, leverage):
    total_exposure = principal * leverage
    leveraged_drawdown = max_drawdown * leverage
    max_possible_loss = principal * abs(leveraged_drawdown)
    liquidation_warning = principal * 0.8
    return total_exposure, leveraged_drawdown, max_possible_loss, liquidation_warning

# ==============================
# 8. 完整历史回测模块（来自untitled30.py，已整合）
# ==============================
def run_backtest(assets, start_date, end_date, target_weights_func, use_ief=True):
    print("\n" + "="*60)
    print("📊 开始历史回测（2020年至今）")
    print("="*60)

    _, _, _, returns, full_data = get_yahoo_finance_data(
        assets, start_date=start_date, end_date=end_date
    )

    portfolio_value = 100000
    current_weights = pd.Series([1/len(assets)]*len(assets), index=assets)
    results = []
    rebalance_dates = returns.resample("3M").last().index

    for date in rebalance_dates:
        hist_returns = returns.loc[:date]
        if use_ief:
            stock_cov = hist_returns[STOCKS_ONLY].cov().values * 252
            stock_weights = calculate_risk_parity_weights(stock_cov)
            target_weights = pd.Series(0.0, index=assets)
            target_weights[STOCKS_ONLY] = stock_weights * 0.9
            target_weights['IEF'] = 0.1
        else:
            target_weights = target_weights_func(hist_returns)

        weight_diff = target_weights - current_weights
        trade_value = abs(weight_diff * portfolio_value)
        total_cost = np.sum(trade_value) * (COMMISSION_RATE + SLIPPAGE_RATE)

        current_weights = target_weights
        portfolio_value -= total_cost

        period_daily_returns = returns.loc[date:].iloc[:63]
        if len(period_daily_returns) > 0:
            strat_return = period_daily_returns.dot(current_weights).mean()
            portfolio_value *= (1 + strat_return)

        results.append({
            "Date": date,
            "Portfolio_Value": round(portfolio_value, 2),
            "Rebalance_Cost": round(total_cost, 2)
        })

    backtest_df = pd.DataFrame(results).set_index("Date")

    if len(backtest_df) > 1:
        daily_ret = backtest_df["Portfolio_Value"].pct_change().dropna()
        final_value = backtest_df["Portfolio_Value"].iloc[-1]
        total_return = (final_value / 100000 - 1) * 100
        sharpe = (daily_ret.mean() / daily_ret.std()) * np.sqrt(252) if len(daily_ret) > 0 else 0
        max_dd = ((backtest_df["Portfolio_Value"] / backtest_df["Portfolio_Value"].cummax()) - 1).min() * 100
        total_cost = backtest_df["Rebalance_Cost"].sum()

        print(f"\n=== 回测结果 ===")
        print(f"初始本金：100,000 美元")
        print(f"最终资产价值：{final_value:,.0f} 美元")
        print(f"总报酬率：{total_return:.2f}%")
        print(f"Sharpe 比率：{sharpe:.3f}")
        print(f"最大回撤：{max_dd:.2f}%")
        print(f"总交易成本：{total_cost:,.0f} 美元")

    return backtest_df

# ==============================
# 9. Plotly权重对比图（修复：确保数据格式正确）
# ==============================
def plot_weight_comparison(current_weights, target_weights, assets):
    fig = go.Figure()

    # 确保都是pandas Series，索引与资产一致
    if not isinstance(current_weights, pd.Series):
        current_weights = pd.Series(current_weights, index=assets)
    if not isinstance(target_weights, pd.Series):
        target_weights = pd.Series(target_weights, index=assets)

    fig.add_bar(
        x=current_weights.index,
        y=current_weights.values,
        name="当前权重",
        marker_color="#FF6B6B"
    )
    fig.add_bar(
        x=target_weights.index,
        y=target_weights.values,
        name="目标权重",
        marker_color="#4ECDC4"
    )

    fig.update_layout(
        barmode='group',
        title="当前权重 vs 目标权重",
        xaxis_title="资产",
        yaxis_title="权重",
        height=500,
        yaxis_tickformat='.1%',  # 显示百分比格式
        hovermode='x unified'
    )
    fig.write_html("weight_comparison.html", auto_open=True)
    return fig
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# ==================== 核心函数（内置） ====================
def rebalance_portfolio(current_weights, target_weights, portfolio_value, cost_rate=0.001):
    weight_diff = target_weights - current_weights
    trade_value = weight_diff * portfolio_value
    total_trade_amount = np.sum(np.abs(trade_value))
    total_cost = total_trade_amount * cost_rate
    cost_ratio = total_cost / portfolio_value
    new_value = portfolio_value - total_cost

    suggested_trades = []
    for ticker, diff in trade_value.items():
        if abs(diff) < 100:
            continue
        if diff < 0:
            suggested_trades.append(f"【賣出】{ticker} : {abs(round(diff,2))} USD")
        else:
            suggested_trades.append(f"【買入】{ticker} : {round(diff,2)} USD")

    return {
        "weight_diff": weight_diff,
        "trade_value": trade_value,
        "estimated_cost": total_cost,
        "cost_ratio": cost_ratio,
        "new_portfolio_value": new_value,
        "suggested_trades": suggested_trades
    }

# ==================== 工具參數設定（已替換為AAPL/MSFT/JNJ/PG/JPM/XOM） ====================
TICKERS = ["AAPL", "MSFT", "JNJ", "PG", "JPM", "XOM"]
total_value = 100000  # 總資產（美元）
transaction_cost = 0.001  # 0.1%交易成本

# 目標權重（同上）
target_weights = pd.Series({
    "AAPL": 0.20,
    "MSFT": 0.20,
    "JNJ":  0.15,
    "PG":   0.15,
    "JPM":  0.15,
    "XOM":  0.15
})

# 模擬目前持倉（刻意製造偏差）
current_holdings = pd.Series({
    "AAPL": 25000,
    "MSFT": 18000,
    "JNJ":  12000,
    "PG":   16000,
    "JPM":  17000,
    "XOM":  12000
})

current_weights = current_holdings / current_holdings.sum()

# ==================== 执行再平衡 ====================
result = rebalance_portfolio(current_weights, target_weights, total_value, transaction_cost)

# ==================== 输出結果（完全符合專案「可操作決策輸出」要求） ====================


# 互動式圖表（Streamlit / Jupyter 皆可顯示）
fig = go.Figure()
fig.show()
# ==============================
# 10. 交易成本敏感度测试（来自untitled30.py）
# ==============================
def run_cost_sensitivity(assets, start_date, end_date, use_ief=True):
    print("\n" + "="*60)
    print("💰 交易成本敏感度测试")
    print("="*60)

    costs = [0.0, 0.001, 0.005]
    results_cost = []

    for c in costs:
        _, _, _, returns, _ = get_yahoo_finance_data(
            assets, start_date=start_date, end_date=end_date
        )

        portfolio_value = 100000
        current_weights = pd.Series([1/len(assets)]*len(assets), index=assets)
        rebalance_dates = returns.resample("3M").last().index

        for date in rebalance_dates:
            hist_returns = returns.loc[:date]
            if use_ief:
                stock_cov = hist_returns[STOCKS_ONLY].cov().values * 252
                stock_weights = calculate_risk_parity_weights(stock_cov)
                target_weights = pd.Series(0.0, index=assets)
                target_weights[STOCKS_ONLY] = stock_weights * 0.9
                target_weights['IEF'] = 0.1
            else:
                target_weights = FIXED_WEIGHTS_STOCKS

            weight_diff = target_weights - current_weights
            trade_value = abs(weight_diff * portfolio_value)
            total_cost = np.sum(trade_value) * c

            current_weights = target_weights
            portfolio_value -= total_cost

            period_daily_returns = returns.loc[date:].iloc[:63]
            if len(period_daily_returns) > 0:
                strat_return = period_daily_returns.dot(current_weights).mean()
                portfolio_value *= (1 + strat_return)

        results_cost.append((c*100, round(portfolio_value, 0)))

    print("\n交易成本敏感度结果：")
    for cost_pct, value in results_cost:
        print(f"交易成本 {cost_pct}% → 最终资产 {value:,.0f} USD")

# ==============================
# 主交互函数
# ==============================
def run_strategy(
    strategy_mode='风险平价+IEF',
    risk_mode='保守型',
    principal=1000000,
    leverage=1,
    run_backtest_flag=False,
    run_cost_sensitivity_flag=False
):
    try:
        # 确定资产池
        if 'IEF' in strategy_mode:
            assets = STOCKS_PLUS_IEF
            use_ief = True
        else:
            assets = STOCKS_ONLY
            use_ief = False

        # 获取真实数据
        prices, vix, cov_matrix, returns, _ = get_yahoo_finance_data(assets)

        # 加载风险模式配置
        cfg = MODES[risk_mode]
        threshold = dynamic_threshold(cfg['base_threshold'], vix)

        # 计算目标权重
        if strategy_mode == '风险平价+IEF':
            stock_cov = cov_matrix[:STOCK_COUNT, :STOCK_COUNT]
            stock_weights = calculate_risk_parity_weights(stock_cov)
            target_weights = np.zeros(len(assets))
            target_weights[:STOCK_COUNT] = stock_weights * STOCK_WEIGHT
            target_weights[STOCK_COUNT] = BOND_WEIGHT
        elif strategy_mode == '风险平价（纯股票）':
            target_weights = calculate_risk_parity_weights(cov_matrix)
        elif strategy_mode == '固定权重（纯股票）':
            target_weights = FIXED_WEIGHTS_STOCKS.values

        target_weights = target_weights / np.sum(target_weights)

        # 计算当前持仓（修复：固定权重模式下根据本金缩放）
        if strategy_mode == '固定权重（纯股票）':
            # 原始持仓比例（总和100000美元）
            original_holdings = pd.Series({
                "AAPL": 25000, "MSFT": 18000, "JNJ": 12000,
                "PG": 16000, "JPM": 17000, "XOM": 12000
            })
            # 根据用户输入的本金缩放
            scaling_factor = principal / original_holdings.sum()
            current_holdings_values = (original_holdings * scaling_factor).values
            holdings = current_holdings_values / prices
        else:
            holdings = np.array([
                int(principal * target_weights[i] / prices[i])
                for i in range(len(assets))
            ])

        # --------------------
        # 5年收益曲线
        # --------------------
        print("\n" + "="*60)
        print("📈 5年资产增长模拟")
        print("="*60)
        print(f"策略模式：{strategy_mode}")
        print(f"风险模式：{risk_mode} - {cfg['description']}")
        print(f"年化收益：{cfg['annual_return']*100:.2f}%")
        print(f"历史最大回撤：{cfg['max_drawdown']*100:.2f}%")
        print(f"动态再平衡阈值：±{threshold*100:.2f}%")

        months, values = simulate_5year(principal, cfg['annual_return'], leverage)
        plt.figure(figsize=(12, 5))
        plt.plot(months, values, color='#1f77b4', linewidth=2.5)
        plt.title('5-year asset value curve (monthly compound interest)）', fontsize=14)
        plt.xlabel('month', fontsize=12)
        plt.ylabel('USD', fontsize=12)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f'✅ 5年后预计总资产：${values[-1]:,.0f}\n')

        # --------------------
        # 杠杆风险分析
        # --------------------
        print("="*60)
        print("⚠️ 杠杆爆仓风险分析")
        print("="*60)
        exposure, dd_lev, loss, warn = calculate_liquidation_risk(
            principal, cfg['max_drawdown'], leverage
        )
        print(f'总持仓暴露：${exposure:,.0f}')
        print(f'杠杆后最大回撤：{dd_lev*100:.2f}%')
        print(f'极端情况最大亏损：${loss:,.0f}')
        print(f'券商爆仓预警线（20%保证金）：${warn:,.0f}')

        if loss > principal * 0.8:
            print('🔴 风险极高：接近爆仓，绝对不可行')
        elif loss > principal * 0.5:
            print('🟡 风险偏高：不建议使用该杠杆')
        else:
            print('🟢 风险可控')
        print()

        # --------------------
        # 再平衡调仓建议
        # --------------------
        print("="*60)
        print("🧾 再平衡调仓建议")
        print("="*60)
        trades, current_weights, target_weights_series, total_value = calc_rebalance(
            prices, holdings, target_weights, threshold, assets
        )
        trading_cost = calculate_trading_costs(trades)

        if not trades:
            print("✅ 当前仓位合理，无需调仓")
            print(f"交易成本：$0.00")
        else:
            for asset, amt in trades.items():
                direction = "买入" if amt > 0 else "卖出"
                print(f'{asset}: {direction} ${abs(amt):,.2f}')
            print(f"\n💰 预计总交易成本：${trading_cost:.2f}")

        # --------------------
        # 目标权重展示
        # --------------------
        print("\n" + "="*60)
        print("📊 目标权重")
        print("="*60)
        weight_df = pd.DataFrame({
            '资产': assets,
            '最新价格': [f"${p:.2f}" for p in prices],
            '当前权重': [f"{w*100:.2f}%" for w in current_weights],
            '目标权重': [f"{w*100:.2f}%" for w in target_weights_series]
        })
        print(weight_df.to_string(index=False))
        print(f"\n权重总和：{np.sum(target_weights)*100:.2f}%")
        print()

        # --------------------
        # Plotly权重对比图（现在可以正常显示了）
        # --------------------
        print("="*60)
        print("📊 权重对比图")
        print("="*60)
        fig = plot_weight_comparison(current_weights, target_weights_series, assets)
        fig.show()

        # --------------------
        # 历史回测（可选）
        # --------------------
        if run_backtest_flag:
            def fixed_weight_func(returns):
                return FIXED_WEIGHTS_STOCKS
            run_backtest(
                assets,
                start_date="2020-01-01",
                end_date=datetime.today().strftime('%Y-%m-%d'),
                target_weights_func=fixed_weight_func,
                use_ief=use_ief
            )

        # --------------------
        # 交易成本敏感度测试（可选）
        # --------------------
        if run_cost_sensitivity_flag:
            run_cost_sensitivity(
                assets,
                start_date="2020-01-01",
                end_date=datetime.today().strftime('%Y-%m-%d'),
                use_ief=use_ief
            )

        print("\n"*2)

    except Exception as e:
        print(f"\n❌ 程序运行失败：{str(e)}")
        print("请检查网络连接或稍后重试")

# ==============================
# 交互式界面
# ==============================
print("🚀 整合最终版（修复权重对比图）：美股蓝筹再平衡工具")
print("="*60)
print("双模式支持 | 完整历史回测 | 100%真实数据")
print()

interact(
    run_strategy,
    strategy_mode=Dropdown(
        options=['风险平价+IEF', '风险平价（纯股票）', '固定权重（纯股票）'],
        value='风险平价+IEF',
        description='策略模式'
    ),
    risk_mode=Dropdown(
        options=['保守型', '平衡型', '进取型'],
        value='保守型',
        description='风险模式'
    ),
    principal=IntSlider(
        min=10000,
        max=10000000,
        step=100000,
        value=1000000,
        description='本金(USD)'
    ),
    leverage=IntSlider(
        min=1,
        max=50,
        value=1,
        description='杠杆倍数'
    ),
    run_backtest_flag=Checkbox(
        value=False,
        description='运行历史回测（2020年至今）'
    ),
    run_cost_sensitivity_flag=Checkbox(
        value=False,
        description='运行交易成本敏感度测试'
    )
);

🚀 整合最终版（修复权重对比图）：美股蓝筹再平衡工具
双模式支持 | 完整历史回测 | 100%真实数据



interactive(children=(Dropdown(description='策略模式', options=('风险平价+IEF', '风险平价（纯股票）', '固定权重（纯股票）'), value='风险平价…